# EXP-4 — Multi Query Retrieval

The LLM generates multiple phrasings of the same question, retrieves separately for each, and deduplicates results. Increases recall at the cost of more LLM calls per query.

## Setup

In [21]:
import os
import time
import sys
import mlflow
import pandas as pd
from dotenv import load_dotenv
from datasets import Dataset

load_dotenv()

# using the new-era langchain packages, not the old monolith
from langchain_community.document_loaders import DirectoryLoader,TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
import warnings
warnings.filterwarnings('ignore')

print("all imports loaded")


all imports loaded


In [2]:
sys.path.append(os.path.abspath(".."))
from data.policies.qa_dataset import dataset

In [3]:
# langsmith traces every chain.invoke() automatically once these env vars are set
# you will see each run in your langsmith project dashboard in real time
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "HR-RAG-Experiments"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "")

print("langsmith tracing is active")


langsmith tracing is active


In [4]:
# set MLFLOW_TRACKING_URI in your .env to point to a remote mlflow server
# if running locally first: mlflow server --host 0.0.0.0 --port 5000
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("HR_RAG_Experiments")

print(f"mlflow tracking uri: {MLFLOW_TRACKING_URI}")


mlflow tracking uri: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow


## Load and Prepare Data

In [5]:
# adjust the path and glob pattern to match your folder structure
loader = DirectoryLoader("../data/policies", glob="**/*.md", loader_cls=TextLoader)
docs = loader.load()

print(f"loaded {len(docs)} documents")


loaded 5 documents


In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(docs)
print(f"total chunks: {len(chunks)}")


total chunks: 98


In [7]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
db = FAISS.from_documents(chunks, embeddings)

print("faiss vector store ready")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4845.43it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


faiss vector store ready


In [8]:

print(f"eval set: {len(dataset['question'])} question-answer pairs")

eval set: 10 question-answer pairs


In [9]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant"
)

In [10]:
prompt = ChatPromptTemplate.from_template("""
You are an HR assistant. Use only the context below to answer the question.
If the answer is not present in the context, say you do not know.

Context:
{context}

Question: {question}

Answer:""")


## Helper Functions

In [11]:
def format_docs(docs):
    # joins retrieved chunks into a single string that goes into the prompt context
    return "\n\n".join(doc.page_content for doc in docs)

In [12]:
from ragas.llms import LangchainLLMWrapper
from ragas.run_config import RunConfig

ragas_llm = LangchainLLMWrapper(llm, run_config=RunConfig(max_workers=1))

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, context_recall
from ragas.run_config import RunConfig

def evaluate_rag(chain, get_docs_fn, dataset):
    answers = []
    contexts = []

    for question in dataset["question"]:
        try:
            # Generate answer
            answer = chain.invoke(question)

            # Retrieve docs (limit to top 3)
            docs = get_docs_fn(question)[:3]

            # Store results (truncate context to avoid timeout)
            answers.append(str(answer))
            contexts.append([str(d.page_content)[:1000] for d in docs])

        except Exception as e:
            print(f"error on: {question[:50]} => {e}")
            answers.append("error")
            contexts.append(["error"])

    # Create evaluation dataset
    eval_dataset = Dataset.from_dict({
        "question": dataset["question"],
        "answer": answers,
        "contexts": contexts,
        "ground_truth": dataset["answer"],
    })

    # Run config (increased timeout)
    run_cfg = RunConfig(
        max_workers=1,
        timeout=300
    )

    # Evaluate (removed heavy metric: context_precision)
    result = evaluate(
        eval_dataset,
        metrics=[faithfulness, context_recall],
        llm=ragas_llm,
        embeddings=embeddings,
        run_config=run_cfg,
        raise_exceptions=False,
    )

    return result

In [14]:
def measure_latency(chain, test_q="What is the leave policy?"):
    start = time.time()
    chain.invoke(test_q)
    return round(time.time() - start, 3)


In [15]:
def log_to_mlflow(run_name, result, latency, retriever_type, top_k=3, extra_params=None):
    with mlflow.start_run(run_name=run_name):
        if hasattr(result, "to_pandas"):
           
            scores = result.to_pandas().mean(numeric_only=True).to_dict()
        else:
            scores = dict(result)

        for k, v in scores.items():
            try:
                mlflow.log_metric(k, float(v))
            except Exception:
                pass

        mlflow.log_metric("latency_seconds", latency)
        mlflow.log_param("retriever_type", retriever_type)
        mlflow.log_param("top_k", top_k)

        if extra_params:
            for k, v in extra_params.items():
                mlflow.log_param(k, str(v))

## Experiment-Specific Imports

In [16]:
from langchain.retrievers.multi_query import MultiQueryRetriever


## Experiment

In [17]:
# multi query retriever handles rephrasing internally
# set verbose=True if you want to see the generated sub-queries in stdout
multi_retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    llm=llm,
)

chain = (
    {"context": multi_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(chain.invoke("What is the leave policy?"))


The Leave Management Policy outlines the types of leaves available to all full-time, part-time, and contractual employees of TechCorp India Pvt. Ltd. The policy applies to all employees across all departments, levels, and locations.

The types of leaves include:

1. Leave Without Pay (LWP)
2. Casual Leave
3. Sick Leave
4. Earned Leave
5. Compensatory Off (Comp-Off)
6. Work From Home (WFH) Leave

The leave application process includes:

1. Minimum application: 3 days minimum per application
2. Maximum consecutive days: 30 days at a stretch (with approval)
3. Carry forward: Up to 30 days per year (maximum accumulation: 90 days) for Earned Leave
4. Encashment: Allowed during service (maximum 15 days per year) and at full & final settlement

Leave encashment rules are as follows:

| Leave Type | During Service | At Separation |
|-----------|---------------|---------------|
| Casual Leave | Not allowed | Not allowed |
| Sick Leave | Not allowed | 50% of balance |
| Earned Leave | Max 15 day

In [18]:
result = evaluate_rag(chain, multi_retriever.invoke, dataset)
latency = measure_latency(chain)

Evaluating: 100%|██████████| 20/20 [08:20<00:00, 25.02s/it]


In [19]:
result,latency

({'faithfulness': 0.2667, 'context_recall': 0.2431}, 4.704)

In [20]:
log_to_mlflow(
    run_name="EXP-4-MULTI-QUERY",
    result=result,
    latency=latency,
    retriever_type="multi-query",
    top_k=3,
)


🏃 View run EXP-4-MULTI-QUERY at: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow/#/experiments/1/runs/fc17cbae74124d69b26f3de5050e3081
🧪 View experiment at: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow/#/experiments/1
